# Karoline Jyske – explore include / exclude

Same job as `02_explore_jyske.ipynb`, for Karoline's Mit Jyske PDF.

Her export is **Danish** (`Dato` / `Beløb`, amounts like `-10.415,00`) and this account is **not** bills-only: most card spend lives here (groceries, ice hockey, shopping). Mehdi's current allow-list would keep almost none of it.

Tweak `bucket()` / `dash_cat()` below, then re-run from that cell.

In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

from jyske_processing import _extract_pdf_text, _expense_category_for, _is_jyske_refund

pd.set_option("display.max_rows", 250)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PDF_PATH = Path(
    "/Users/mehdiordikhani/Library/Mobile Documents/com~apple~CloudDocs/faimly_files/Karoline Doser_2023-06-30-2026-08-30.pdf"
)
print("pdf_path:", PDF_PATH)
print("exists:", PDF_PATH.exists())

pdf_path: /Users/mehdiordikhani/Library/Mobile Documents/com~apple~CloudDocs/faimly_files/Karoline Doser_2023-06-30-2026-08-30.pdf
exists: True


## 1. Load Danish PDF

Unlike Mehdi's English CSV, this file has no `MainCategory` / `Category`. We only get date, text, amount, balance.

In [4]:
DK_AMT = r"-?(?:\d{1,3}(?:\.\d{3})+|\d+),\d{2}"
ROW_RE = re.compile(
    rf"^(?P<date>\d{{2}}\.\d{{2}}\.\d{{4}})\s+(?P<text>.+?)\s+(?P<amount>{DK_AMT})\s+(?P<balance>{DK_AMT})\s*$"
)


def parse_dk_amount(value) -> float:
    s = str(value).strip().replace("\xa0", "").replace(" ", "")
    if not s:
        return np.nan
    negative = s.startswith("-")
    s = s.lstrip("-").replace(".", "").replace(",", ".")
    n = float(s)
    return -n if negative else n


text = _extract_pdf_text(PDF_PATH)
rows = []
for line in text.splitlines():
    line = line.strip()
    if not line or line.lower().startswith("side ") or line.startswith("Dokument"):
        continue
    m = ROW_RE.match(line)
    if not m:
        continue
    rows.append(
        {
            "date": pd.to_datetime(m.group("date"), dayfirst=True),
            "text": m.group("text").strip(),
            "amount": parse_dk_amount(m.group("amount")),
            "balance": parse_dk_amount(m.group("balance")),
        }
    )

df = pd.DataFrame(rows)
df["month"] = df["date"].dt.to_period("M").astype(str)
print("rows:", len(df), "| unique texts:", df["text"].nunique())
print("date range:", df["date"].min().date(), "→", df["date"].max().date())
print("outflows:", int((df["amount"] < 0).sum()), f"{df.loc[df['amount'] < 0, 'amount'].sum():,.2f}")
print("inflows:", int((df["amount"] > 0).sum()), f"{df.loc[df['amount'] > 0, 'amount'].sum():,.2f}")
print("net:", f"{df['amount'].sum():,.2f}")
df.head(8)

rows: 2721 | unique texts: 823
date range: 2023-06-30 → 2026-08-28
outflows: 2558 -1,515,039.31
inflows: 163 1,741,084.61
net: 226,045.30


,date,text,amount,balance,month
0,2026-08-28,VD NOVO NORDISK VTC 653310,-36.00,"265,881.81",2026-08
1,2026-08-28,VD ROEDOVRE SKOEJTE ISHO,-30.00,"265,917.81",2026-08
2,2026-08-28,VD NOVO NORDISK VTC 653310,-19.00,"265,947.81",2026-08
3,2026-08-27,VD 7-ELEVEN 090,-124.00,"265,966.81",2026-08
4,2026-08-27,VD EMMERYS 510,-62.00,"266,090.81",2026-08
5,2026-08-27,VD BREW PUB,-60.00,"266,152.81",2026-08
6,2026-08-26,VD AMAZON.DE,-37.68,"266,212.81",2026-08
7,2026-08-25,VD JOE THE JUICE A/S,-584.00,"266,250.49",2026-08


## 2. What Mehdi's current parser would keep

Almost nothing — her recurring bills are different names (kommune, school, energy, Djøf).

In [5]:
df["mehdi_cat"] = df["text"].map(_expense_category_for)
df["mehdi_refund"] = df["text"].map(_is_jyske_refund)
keep = df["mehdi_cat"].notna() | df["mehdi_refund"]
print("would keep:", int(keep.sum()), "of", len(df))
(
    df.loc[keep]
    .assign(cat=df["mehdi_cat"].fillna("refund"))
    .groupby("cat")["amount"]
    .agg(n="count", sum="sum")
)

would keep: 5 of 2721


,n,sum
cat,,
Bank fees,3,-8.00
Health,2,-346.50


## 3. Group by text

This is the real filter surface — 800+ distinct descriptions.

In [6]:
by_text = (
    df.groupby("text")["amount"]
    .agg(n="count", sum="sum")
    .assign(abs_sum=lambda x: x["sum"].abs())
    .sort_values("abs_sum", ascending=False)
)
by_text.head(60)

,n,sum,abs_sum
text,,,
Lønoverførsel,40,"1,418,254.76","1,418,254.76"
"Til Rentegaranti 3,25% 10 2024",3,"-150,000.00","150,000.00"
"Lånesagskonto, deponering",2,"-130,000.00","130,000.00"
BS RØDOVRE KOMMUNE,34,"-129,749.50","129,749.50"
MobilePay Mehdi Ordikhani Seye,29,"-119,013.65","119,013.65"
BS EJERFORENINGEN PARKKANTEN,37,"-105,425.03","105,425.03"
BS SANKT PETRI SKOLE,21,"-101,215.00","101,215.00"
Overførsel,3,"95,700.00","95,700.00"
Lara - Oskarkonto,5,"-52,144.00","52,144.00"


## 4. Proposed buckets — **edit this cell**

| Bucket | Meaning | Examples |
|---|---|---|
| `exclude_savings_invest` | Savings / kids accounts / Nordnet / deposit | `Opsparingskonto`, `Oskarkonto`, `Nordnet`, `Lånesagskonto` |
| `exclude_internal` | Transfers to Mehdi / own accounts | `MobilePay Mehdi…`, `Overførsel`, account numbers |
| `income` | Salary and benefits | `Lønoverførsel`, `Børne- og Ungeydelse`, feriepenge |
| `expense_bill` | Recurring bills paid from Jyske | kommune, ejerforening, school, energy, unions, insurance |
| `review_card_or_other` | Daily card spend — include only if we treat this account like Revolut | Rema, Netto, hockey, Uniqlo, Novo canteen |

In [7]:
def bucket(text: str) -> str:
    t = str(text).casefold()
    # Income first: "Lønoverførsel" contains "overførsel".
    if t.startswith("lønoverførsel") or "børne- og ungeydelse" in t or "feriepenge" in t or "overskydende skat" in t:
        return "income"
    if t.startswith("doser, maritta") or "universitetet i oslo" in t:
        return "income"
    if any(x in t for x in ("opsparing", "oskar", "rentegaranti", "nordnet", "lånesagskonto")):
        return "exclude_savings_invest"
    if any(x in t for x in ("mehdi", "5030 1360462", "karoline doser")):
        return "exclude_internal"
    if t == "overførsel" or t.startswith("overførsel "):
        return "exclude_internal"
    if t.replace(" ", "").startswith("6695") or t.startswith("til 6695") or t.startswith("salary august"):
        return "exclude_internal"
    if any(
        x in t
        for x in (
            "rødovre kommune",
            "roedovre kommune",
            "ejerforeningen parkkanten",
            "sankt petri",
            "andel energi",
            "akademikernes",
            "djøf",
            "sygeforsikringen danmark",
            "lb forsikring",
            "pure gym",
            "personskatter",
            "ejendomsskattelån",
            "omkostninger, netbank",
            "dankort, årlig",
        )
    ):
        return "expense_bill"
    return "review_card_or_other"


def dash_cat(text: str, bkt: str) -> str:
    t = str(text).casefold()
    if bkt == "expense_bill":
        if "sankt petri" in t:
            return "School"
        if "ejerforening" in t or "ejendomsskat" in t:
            return "Home"
        if "kommune" in t:
            return "Home tax"
        if "andel energi" in t:
            return "Utilities"
        if "akademikernes" in t or "djøf" in t:
            return "Unions"
        if "sygeforsikring" in t or "lb forsikring" in t:
            return "Insurance"
        if "pure gym" in t:
            return "Health"
        if "personskatter" in t:
            return "Tax"
        if "omkostninger" in t or "dankort" in t:
            return "Bank fees"
        return "Other bill"
    if bkt != "review_card_or_other":
        return ""
    if any(x in t for x in ("rema", "netto", "foetex", "føtex", "lidl", "aldi", "7-eleven")):
        return "Groceries"
    if any(x in t for x in ("skoejte", "hockey", "skatertown", "holdsport", "mighty bulls", "gladsaxe")):
        return "Ice Hockey"
    if any(x in t for x in ("uniqlo", "hm dk", "name it", "kids coolshop", "pop mart")):
        return "Shopping"
    if any(x in t for x in ("emmerys", "joe  the juice", "olea", "brew pub")):
        return "Eat Out"
    if any(x in t for x in ("novo nordisk vtc", "parkman", "rejsekort", "easyjet")):
        return "Transport"
    if any(x in t for x in ("netflix", "disney", "amazon prim", "amznprime", "lebara")):
        return "Streaming / phone"
    if "amazon" in t or "amzn" in t:
        return "Shopping"
    if "tandlæge" in t or "tandlaege" in t:
        return "Health"
    return "Other / review"


df["bucket"] = df["text"].map(bucket)
df["dash"] = [dash_cat(t, b) for t, b in zip(df["text"], df["bucket"])]
df["bucket"].value_counts()

bucket
review_card_or_other      2356
expense_bill               198
exclude_internal            74
income                      58
exclude_savings_invest      35
Name: count, dtype: int64

## 5. Totals under that proposal

In [8]:
(
    df.groupby("bucket")["amount"]
    .agg(n="count", sum="sum")
    .sort_values("sum")
)

,n,sum
bucket,,
expense_bill,198,"-452,500.69"
exclude_savings_invest,35,"-420,047.00"
review_card_or_other,2356,"-374,932.31"
exclude_internal,74,"-80,057.48"
income,58,"1,553,582.78"


In [9]:
print("=== Recurring bills (clear include) ===")
bills = df[df["bucket"].eq("expense_bill")]
display(
    bills.groupby(["dash", "text"])["amount"]
    .agg(n="count", sum="sum")
    .sort_values("sum")
)

=== Recurring bills (clear include) ===


n         sum
dash      text                                              
Home tax  BS RØDOVRE KOMMUNE                  34 -129,749.50
Home      BS EJERFORENINGEN PARKKANTEN        37 -105,425.03
School    BS SANKT PETRI SKOLE                21 -101,215.00
          Til Sankt Petri Skole                4  -20,225.00
Unions    BS AKADEMIKERNES A-KASSE            13  -19,251.00
Tax       DK SKTST/Personskatter               3  -17,256.00
Unions    MobilePay Djøf                      13  -13,327.00
Utilities BS ANDEL ENERGI A/S                 27  -12,784.83
Insurance BS SYGEFORSIKRINGEN DANMARK         13  -11,963.00
          BS LB FORSIKRING A/S                 3   -7,503.36
Utilities Til Andel Energi A/S                 8   -5,318.53
Home      DK SKTST/Ejendomsskattelån           1   -4,803.12
Home tax  MobilePay RØDOVRE KOMMUNE            1   -2,855.82
School    Til Sankt petri                      1   -2,100.00
          MobilePay Sankt Petri Skolefor       4     -460.00
          MobilePay Sankt Petri Musiksko       1     -350.00
Health    BS PURE GYM DENMARK A/S              2     -346.50
School    MobilePay Sankt Petri Skole          4     -230.00
Bank fees 1 Dankort, årlig kortbetaling        1     -200.00
School    MobilePay Sankt Petri Kirke          1     -150.00
          MobilePay Sankt Petri SFO            1      -50.00
Home tax  Til Rødovre Kommune                  1      -20.00
Bank fees Omkostninger, Netbank og Mobilbank   3       -8.00
Home tax  Rødovre Kommune                      1    3,091.00

In [12]:
bills

,date,text,amount,balance,month,mehdi_cat,mehdi_refund,bucket,dash
9,2026-08-25,Rødovre Kommune,"3,091.00","267,934.49",2026-08,None,False,expense_bill,Home tax
29,2026-08-03,MobilePay Djøf,"-1,123.00","233,863.54",2026-08,None,False,expense_bill,Unions
31,2026-08-03,BS SANKT PETRI SKOLE,"-10,415.00","235,786.54",2026-08,None,False,expense_bill,School
32,2026-08-03,BS EJERFORENINGEN PARKKANTEN,"-3,091.85","246,201.54",2026-08,None,False,expense_bill,Home
33,2026-08-03,BS ANDEL ENERGI A/S,-418.21,"249,293.39",2026-08,None,False,expense_bill,Utilities
34,2026-08-03,BS RØDOVRE KOMMUNE,"-3,091.00","249,711.60",2026-08,None,False,expense_bill,Home tax
63,2026-07-06,BS SYGEFORSIKRINGEN DANMARK,-959.00,"230,955.50",2026-07,None,False,expense_bill,Insurance
67,2026-07-01,BS SANKT PETRI SKOLE,"-2,525.00","232,049.80",2026-07,None,False,expense_bill,School
68,2026-07-01,BS EJERFORENINGEN PARKKANTEN,"-3,091.85","234,574.80",2026-07,None,False,expense_bill,Home
69,2026-07-01,BS ANDEL ENERGI A/S,-384.20,"237,666.65",2026-07,None,False,expense_bill,Utilities


In [10]:
print("=== Suggested dashboard categories if we also include card spend ===")
(
    df[df["dash"].ne("")]
    .groupby("dash")["amount"]
    .agg(n="count", sum="sum")
    .sort_values("sum")
)

=== Suggested dashboard categories if we also include card spend ===


,n,sum
dash,,
Other / review,1286,"-228,842.81"
Home tax,37,"-129,534.32"
School,37,"-124,780.00"
Home,38,"-110,228.15"
Ice Hockey,243,"-44,382.44"
Groceries,227,"-37,823.03"
Unions,26,"-32,578.00"
Shopping,84,"-25,519.83"
Transport,373,"-20,241.83"


In [11]:
def show(bkt: str, n: int = 20):
    sub = df[df["bucket"].eq(bkt)]
    print(f"=== {bkt}  n={len(sub)}  sum={sub['amount'].sum():,.2f} ===")
    return (
        sub.groupby("text")["amount"]
        .agg(n="count", sum="sum")
        .assign(abs_sum=lambda x: x["sum"].abs())
        .sort_values("abs_sum", ascending=False)
        .head(n)
    )

show("review_card_or_other", 40)

=== review_card_or_other  n=2356  sum=-374,932.31 ===


,n,sum,abs_sum
text,,,
Maritta Doser,1,"-22,443.30","22,443.30"
VD REMA 1000 ROEDOVRE,77,"-16,866.27","16,866.27"
VD THOMANN DE DK,2,"-15,139.00","15,139.00"
VD NETTO KAFFEVEJ ROEDOVRE 7,57,"-10,109.18","10,109.18"
Next 50 of 20k dkk,1,"10,000.00","10,000.00"
VD Pension Sportalm,1,"-8,268.99","8,268.99"
VD NOVO NORDISK VTC 653310,328,"-8,145.00","8,145.00"
VD RESIDENCE MY SCILIAR,1,"-7,783.94","7,783.94"
VD SKATERTOWN APS,15,"-7,618.20","7,618.20"
